# 2. Renovate and count materials

Replace windows and add wall insulation. One renovate call groups the work into one step; two successive calls record two steps. This notebook calculates material impacts only. The next notebook adds actual annual energy.

In [ ]:
from dataclasses import replace
from math import isclose
from pathlib import Path
from uuid import uuid4

from renovation_case import make_case

from epsm import RenovationResult, assess_renovation

case = make_case()
settings = replace(case["settings"], include_energy=False)
single = assess_renovation(case["single_step"], settings)
staged = assess_renovation(case["step_2"], settings)
for row in staged.table():
    print(row)
assert single.complete and staged.complete
assert isclose(single.stages[-1].total_kgco2e, staged.stages[-1].total_kgco2e)

Step 0 is the baseline. Existing materials are retained without charging again for their original construction. Material impact this step counts additions and declared treatment of removed materials. Material impact so far adds the material work through that step. A5.1 is installation waste, not complete installation impact. Boverket supplies the selected A1–A3, A4 and A5.1 values. Window removal factors are explicit teaching assumptions.

In [ ]:
print("Added materials:", staged.quantities(action="added", by_material=True))
print("Removed materials:", staged.quantities(action="removed", by_material=True))
assert staged.stages[0].removed_quantities
assert all(row["mass_kg"] is None for row in staged.stages[0].removed_quantities)

Missing information must stay visible. Remove one required window-treatment factor: the quantities are still available, while the affected material total becomes unavailable.

In [ ]:
factors = {
    identity: dict(values) for identity, values in settings.material_factors.items()
}
removed_id = staged.stages[0].removed_quantities[0]["material_id"]
del factors[removed_id]["C4"]
incomplete = assess_renovation(
    case["step_2"], replace(settings, material_factors=factors)
)
print(incomplete.stages[0].requirements)
assert incomplete.stages[0].material_kgco2e is None
assert incomplete.stages[0].removed_quantities

Save and reload a complete report. The file keeps the building states, quantities, factors, sources and assessment choices. Saving refuses an existing file by default.

In [ ]:
output = Path("../outputs/notebooks") / uuid4().hex
output.mkdir(parents=True)
path = staged.save(output / "materials.json")
restored = RenovationResult.load(path)
assert restored == staged
figure = restored.plot()
figure